In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Bagging Trees for Direct Marketing Response (UCI Bank Marketing)

### Context & Goal

You’re hired by a retail bank to predict whether a customer will subscribe to a term deposit after a marketing call. Your job is to build a **bagging classifier** that beats a single decision tree and to prove the gain is due to **variance reduction**, not luck.

### Learning Targets

* Engineer a full binary-classification pipeline on mixed tabular data (categorical + numeric).
* Train a single Decision Tree vs a Bagging ensemble; quantify bias/variance behaviour.
* Handle **class imbalance** correctly; report metrics that matter (ROC-AUC, PR-AUC, F1).
* Use cross-validation and (optionally) out-of-bag scoring to estimate generalisation.

### Dataset

* **Name:** Bank Marketing
* **Source URL:** [UCI Machine Learning Repository – Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank-marketing)
* **Description:** Marketing calls to Portuguese bank customers with demographics (age, job, education), call details (duration, contact type, month, day), and prior campaign outcomes.
* **Target:** `y` — `yes` if client subscribed, `no` otherwise.
* **Why this fits:** Mixed types, moderate size (~41k rows), **imbalanced** positive class — ideal to test whether bagging stabilises trees and improves decision quality beyond raw accuracy.

### Task Instructions

1. **Data loading & audit**

   * Load the **full** dataset (`bank-full.csv`). Verify row count, dtypes, missing markers (e.g., `"unknown"`), and class ratio.
   * Create a compact data dictionary from the header; flag high-cardinality categoricals if any.

In [2]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
bank_marketing = fetch_ucirepo(id=222)

# data (as pandas dataframes)
X = bank_marketing.data.features
y = bank_marketing.data.targets

In [3]:
meta = bank_marketing.metadata
meta_df = pd.DataFrame(
	data=[(k, v) for k, v in meta.items() if v is not None],
	columns=["Attribute", "Value"],
)
display(meta_df)

,Attribute,Value
0,uci_id,222
1,name,Bank Marketing
2,repository_url,https://archive.ics.uci.edu/dataset/222/bank+m...
3,data_url,https://archive.ics.uci.edu/static/public/222/...
4,abstract,The data is related with direct marketing camp...
5,area,Business
6,tasks,[Classification]
7,characteristics,[Multivariate]
8,num_instances,45211
9,num_features,16


In [4]:
# Check the data structure and types
X.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN


In [5]:
# Examine data types and missing values
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          45211 non-null  int64 
 1   job          44923 non-null  object
 2   marital      45211 non-null  object
 3   education    43354 non-null  object
 4   default      45211 non-null  object
 5   balance      45211 non-null  int64 
 6   housing      45211 non-null  object
 7   loan         45211 non-null  object
 8   contact      32191 non-null  object
 9   day_of_week  45211 non-null  int64 
 10  month        45211 non-null  object
 11  duration     45211 non-null  int64 
 12  campaign     45211 non-null  int64 
 13  pdays        45211 non-null  int64 
 14  previous     45211 non-null  int64 
 15  poutcome     8252 non-null   object
dtypes: int64(7), object(9)
memory usage: 5.5+ MB


In [6]:
# Check for missing values (NaN)
print("Missing values per column:")
print(X.isnull().sum())
print(f"\nTotal missing values: {X.isnull().sum().sum()}")

Missing values per column:
age                0
job              288
marital            0
education       1857
default            0
balance            0
housing            0
loan               0
contact        13020
day_of_week        0
month              0
duration           0
campaign           0
pdays              0
previous           0
poutcome       36959
dtype: int64

Total missing values: 52124


In [7]:
# Check class distribution of target variable
print("Target variable distribution:")
print(y.value_counts())
print(f"\nClass ratio: {y.value_counts(normalize=True)}")

Target variable distribution:
y  
no     39922
yes     5289
Name: count, dtype: int64

Class ratio: y  
no     0.883015
yes    0.116985
Name: proportion, dtype: float64


2. **Preprocessing choices (justify each)**

   * Encode categoricals (one-hot or target encoding—your call; defend it).
   * Treat `"unknown"` explicitly (keep as level vs impute; explain).
   * Scale numeric features *only if* your downstream choice needs it (trees don’t; pipelines should still be consistent).
   * Split **stratified** into train/test (e.g., 70/30). Fix a seed.


In [8]:
# Define column groups for preprocessing
# Binary columns that need yes/no -> 0/1 conversion
binary = ["default", "housing", "loan"]

# Categorical columns that need one-hot encoding
categorical = [
	"job",
	"marital",
	"education",
	"contact",
	"day_of_week",
	"month",
	"poutcome",
]

# Numeric columns (no transformation needed for trees, but good to identify)
numerical = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]

In [9]:
# Create a copy to avoid modifying the original data
X = X.copy()

# STEP 1: Handle "unknown" values in categorical columns
# Decision: Keep "unknown" as a valid category level
# Justification: "unknown" is informative - it indicates missing/withheld information
# which may correlate with the target (e.g., people who don't share job info)
for col in categorical:
	if col in X.columns:
		# Check if "unknown" exists in this column
		if "unknown" in X[col].values:
			print(
				f"'{col}' has {(X[col] == 'unknown').sum()} 'unknown' values - keeping as category"
			)

In [10]:
# STEP 2: Convert binary yes/no columns to 0/1
# Using simple mapping instead of custom transformer for clarity
# yes -> 1 (positive outcome), no -> 0 (negative outcome)
for col in binary:
	if col in X.columns:
		X[col] = X[col].map({"yes": 1, "no": 0})
		print(f"Converted '{col}': {X[col].value_counts().to_dict()}")

Converted 'default': {0: 44396, 1: 815}
Converted 'housing': {1: 25130, 0: 20081}
Converted 'loan': {0: 37967, 1: 7244}


In [11]:
# STEP 3: Verify data types are appropriate
# Ensure numeric columns are numeric (should already be from the dataset)
print("Data types after binary conversion:")
print(X.dtypes)
print(f"\nShape: {X.shape}")

Data types after binary conversion:
age             int64
job            object
marital        object
education      object
default         int64
balance         int64
housing         int64
loan            int64
contact        object
day_of_week     int64
month          object
duration        int64
campaign        int64
pdays           int64
previous        int64
poutcome       object
dtype: object

Shape: (45211, 16)


In [12]:
# STEP 4: Check if imputation is needed
# Only impute if there are actual NaN values (not "unknown" strings)
if X.isnull().sum().sum() > 0:
	print("Missing values detected - applying imputation")
	from sklearn.impute import SimpleImputer

	# Separate numeric and categorical for different imputation strategies
	numeric_cols = X.select_dtypes(include=[np.number]).columns
	categorical_cols = X.select_dtypes(include=["object"]).columns

	# Impute numerics with median, categoricals with most frequent
	if len(numeric_cols) > 0:
		num_imputer = SimpleImputer(strategy="median")
		X[numeric_cols] = num_imputer.fit_transform(X[numeric_cols])

	if len(categorical_cols) > 0:
		cat_imputer = SimpleImputer(strategy="most_frequent")
		X[categorical_cols] = cat_imputer.fit_transform(X[categorical_cols])

	print(f"After imputation: {X.isnull().sum().sum()} missing values")
else:
	print("No missing values detected - skipping imputation")

Missing values detected - applying imputation
After imputation: 0 missing values
After imputation: 0 missing values


In [13]:
# STEP 5: One-hot encode categorical variables
# Decision: Use one-hot encoding with drop='first' to avoid multicollinearity
# Justification:
# - Trees don't require it, but it makes features interpretable
# - drop='first' avoids dummy variable trap
# - Keeps all categorical information without imposing ordinal relationships

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Create a column transformer that applies OneHotEncoder only to categorical columns
preprocessor = ColumnTransformer(
	transformers=[
		(
			"cat",
			OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
			categorical,
		)
	],
	remainder="passthrough",  # Keep binary and numeric columns as they are
)

# Apply the transformation
X_encoded = preprocessor.fit_transform(X)

# Get feature names after one-hot encoding
feature_names = preprocessor.named_transformers_["cat"].get_feature_names_out(
	categorical
)
remaining_cols = [col for col in X.columns if col not in categorical]
all_feature_names = list(feature_names) + remaining_cols

# Create DataFrame with proper column names
# CRITICAL: Convert back to DataFrame to preserve column names
X_encoded = pd.DataFrame(X_encoded, columns=all_feature_names, index=X.index)

print(f"Original features: {X.shape[1]}")
print(f"After one-hot encoding: {X_encoded.shape[1]}")
print(f"Added {X_encoded.shape[1] - X.shape[1]} dummy variables")

Original features: 16
After one-hot encoding: 67
Added 51 dummy variables


In [14]:
# Verify the original data before encoding
print("Original data (first 5 rows):")
X.head()

Original data (first 5 rows):


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
0,58.0,management,married,tertiary,0.0,2143.0,1.0,0.0,cellular,5.0,may,261.0,1.0,-1.0,0.0,failure
1,44.0,technician,single,secondary,0.0,29.0,1.0,0.0,cellular,5.0,may,151.0,1.0,-1.0,0.0,failure
2,33.0,entrepreneur,married,secondary,0.0,2.0,1.0,1.0,cellular,5.0,may,76.0,1.0,-1.0,0.0,failure
3,47.0,blue-collar,married,secondary,0.0,1506.0,1.0,0.0,cellular,5.0,may,92.0,1.0,-1.0,0.0,failure
4,33.0,blue-collar,single,secondary,0.0,1.0,0.0,0.0,cellular,5.0,may,198.0,1.0,-1.0,0.0,failure


In [15]:
# Verify the encoded data (notice column names are preserved!)
print("Encoded data (first 5 rows):")
print(f"Columns: {list(X_encoded.columns[:10])}... (showing first 10)")
X_encoded.head()

Encoded data (first 5 rows):
Columns: ['job_blue-collar', 'job_entrepreneur', 'job_housemaid', 'job_management', 'job_retired', 'job_self-employed', 'job_services', 'job_student', 'job_technician', 'job_unemployed']... (showing first 10)


,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,...,poutcome_success,age,default,balance,housing,loan,duration,campaign,pdays,previous
0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,58.0,0.0,2143.0,1.0,0.0,261.0,1.0,-1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,44.0,0.0,29.0,1.0,0.0,151.0,1.0,-1.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,33.0,0.0,2.0,1.0,1.0,76.0,1.0,-1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,47.0,0.0,1506.0,1.0,0.0,92.0,1.0,-1.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,33.0,0.0,1.0,0.0,0.0,198.0,1.0,-1.0,0.0


In [17]:
# STEP 6: Stratified train/test split
# Stratification ensures both sets have the same class ratio as the original data
# Critical for imbalanced datasets to avoid biased evaluation
from sklearn.model_selection import train_test_split

# Convert target to binary if it's yes/no
if y.iloc[:, 0].dtype == "object":
	y_binary = y.iloc[:, 0].map({"yes": 1, "no": 0})
else:
	y_binary = y.iloc[:, 0]

X_train, X_test, y_train, y_test = train_test_split(
	X_encoded,
	y_binary,
	test_size=0.3,
	random_state=42,
	stratify=y_binary,  # Ensures same class distribution in train and test
)

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True)}")
print(f"\nTest class distribution:\n{y_test.value_counts(normalize=True)}")

Training set: (31647, 67), Test set: (13564, 67)
Train class distribution:
y
0    0.883022
1    0.116978
Name: proportion, dtype: float64

Test class distribution:
y
0    0.882999
1    0.117001
Name: proportion, dtype: float64


3. **Baseline: single tree**

   * Train a DecisionTreeClassifier. Set depth/leaf constraints you deem reasonable; document them.
   * Evaluate on the **test set** with: ROC-AUC, PR-AUC, accuracy, precision, recall, F1. Include a confusion matrix.
   * Record cross-validated ROC-AUC on the **training split** (e.g., StratifiedKFold, k=5 or 10) to estimate variance (mean ± std).

In [18]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor

param_grid_tree = {
	"criterion": ["squared_error", "absolute_error"],
	"max_depth": [None, 3, 5, 8, 12, 16],
	"min_samples_split": [2, 5, 10, 20],
	"min_samples_leaf": [1, 2, 5, 10, 20],
	"max_features": [None, "sqrt", "log2"],
}

grid_tree = GridSearchCV(
	estimator=DecisionTreeRegressor(random_state=42),
	param_grid=param_grid_tree,
	cv=5,
	scoring="neg_root_mean_squared_error",
	n_jobs=-1,
	verbose=1,
	refit=True,
	return_train_score=True,
	error_score="raise",
)

grid_tree.fit(X_train, y_train)

Fitting 5 folds for each of 720 candidates, totalling 3600 fits


KeyboardInterrupt: 

4. **Bagging ensemble**

   * Train a BaggingClassifier with your decision tree as base estimator. Use bootstrapping and **n_estimators ≥ 50**. Keep other settings explicit (max_samples, max_features).
   * Compute the **same metrics** on the test set.
   * Estimate variability via CV as above; additionally report **OOB score** if you enable `oob_score=True`, and compare OOB vs CV.


5. **Imbalance sensitivity**

   * Re-run (tree and bagging) with a **class-weighting** strategy or a **resampled** training set (e.g., stratified undersample or SMOTE). Report the delta in PR-AUC and recall. Explain which approach you’d ship and why.


6. **Analysis**

   * State clearly whether bagging improved performance. Back it with numbers (ROC-AUC/PR-AUC and the reduction in CV std).
   * Argue **bias vs variance**: did test metrics rise mainly because variance fell (tighter CV distribution) or because you altered bias (systematic error) via reweighting/encoding?


### Evaluation

Success is:

* Reproducible pipeline; decisions justified, not guessed.
* A metric table comparing **Single Tree vs Bagging** (and, if attempted, weighted/resampled variants) with ROC-AUC and PR-AUC as primary.
* Evidence of **variance reduction** (smaller CV std or tighter OOB confidence).
* A short, technical conclusion on deployability (precision–recall trade-off under imbalance).


### Stretch Ideas

* Tune `n_estimators`, `max_samples`, and `bootstrap_features`; plot performance vs ensemble size.
* Compare Bagging with **RandomForest** (feature subsampling) and explain differences in bias/variance.
* Calibrate probabilities (Platt/isotonic) and evaluate **Brier score** + reliability curves.
